# Ollama Model Inference

> Load and run ollama models


In [ ]:
# | default_exp llm


In [ ]:
# | exporti
import dotenv
import json
import os
import httpx

In [ ]:
# | exporti
dotenv.load_dotenv()


In [ ]:
# | exporti
from typing import Literal, Optional
from ollama import list as ollama_list


In [ ]:
# | hide
from nbdev.showdoc import *


In [ ]:
# | export
def list_local_models():
  """List local models."""
  return [m.model for m in ollama_list()["models"]]


In [ ]:
list_local_models()


In [ ]:
# | exporti
from tqdm import tqdm
from ollama import pull


In [ ]:
# | exporti
def _download_model(model_name: str):
  """Download a model from ollama."""
  current_digest, bars = "", {}
  for progress in pull(model_name, stream=True):
    digest = progress.get("digest", "")
    if digest != current_digest and current_digest in bars:
      bars[current_digest].close()

    if not digest:
      print(progress.get("status"))
      continue

    if digest not in bars and (total := progress.get("total")):
      bars[digest] = tqdm(total=total, desc=f"pulling {digest[7:19]}", unit="B", unit_scale=True)

    if completed := progress.get("completed"):
      bars[digest].update(completed - bars[digest].n)

    current_digest = digest


In [ ]:
model_name = "gemma3:27b"
local_models = list_local_models()
if model_name not in local_models:
  _download_model(model_name)


In [ ]:
# | exporti
from ollama import chat as ollama_chat


In [ ]:
ollama_response = ollama_chat(
  model=model_name, think=None, messages=[{"role": "user", "content": "Merhaba. Nasılsın?"}]
)
ollama_response.message.content, ollama_response.message.thinking


In [ ]:
# | export
def create_message(content: Optional[str], role: Literal["user", "assistant", "system"] = "user"):
  """Create a message for the chat."""
  return {"role": role, "content": content}


In [ ]:
# | exporti
import base64
import mimetypes
from pathlib import Path


In [ ]:
# | export
def create_message_with_resource(
  text: str,
  resource_path: Path | str,
  role: Literal["user", "assistant"] = "user",
) -> dict:
  """Create a message with an attached file (image/PDF) for vision models.
  
  Args:
    text: The text prompt to accompany the resource
    resource_path: Path to the file (image or PDF)
    role: Message role (user or assistant)
    
  Returns:
    Message dict in litellm vision format with base64-encoded content
  """
  resource_path = Path(resource_path)
  
  # Read and encode file
  file_bytes = resource_path.read_bytes()
  base64_data = base64.b64encode(file_bytes).decode("utf-8")
  
  # Detect MIME type
  mime_type, _ = mimetypes.guess_type(str(resource_path))
  if mime_type is None:
    # Default based on extension
    ext = resource_path.suffix.lower()
    mime_map = {
      ".pdf": "application/pdf",
      ".png": "image/png",
      ".jpg": "image/jpeg",
      ".jpeg": "image/jpeg",
      ".gif": "image/gif",
      ".webp": "image/webp",
    }
    mime_type = mime_map.get(ext, "application/octet-stream")
  
  # Create message with multimodal content
  return {
    "role": role,
    "content": [
      {"type": "text", "text": text},
      {
        "type": "image_url",
        "image_url": {"url": f"data:{mime_type};base64,{base64_data}"},
      },
    ],
  }


In [ ]:
# | exporti
def _chat_one(model: str, message: str, think: Optional[bool] = None, **kwargs):
  """Chat with the model."""
  return ollama_chat(
    model=model, think=think, messages=[create_message(message)], **kwargs
  ).message.content


In [ ]:
_chat_one("gemma3:27b", "Merhaba. Nasılsın?")


In [ ]:
# | exporti
from pydantic import BaseModel


In [ ]:
# | export
class OutputSchema(BaseModel):
  """Base class for structured LLM outputs. Subclass this to define your schema.

  The optional `content` field will be used for chat history when present."""

  content: str | None = None

In [ ]:
class _Response(OutputSchema):
  """Example response schema."""

  reasoning: str
  is_happy: bool


In [ ]:
_Response.model_validate_json(
  _chat_one(model_name, "Mulu musun?", think=None, format=_Response.model_json_schema())
)


# LLM Cloud


In [ ]:
# | exporti
from ollama import ChatResponse
import litellm
litellm.suppress_debug_info = True  # Stop printing "Provider List" spam
from litellm import completion as litellm_completion
from urllib.request import Request, urlopen


def _schema_name(schema: type[BaseModel]) -> str:
  name = getattr(schema, "__name__", "structured_response")
  return "".join(ch if ch.isalnum() or ch == "_" else "_" for ch in name) or "structured_response"


def _openrouter_response_format(schema: type[BaseModel]) -> dict:
  return {
    "type": "json_schema",
    "json_schema": {
      "name": _schema_name(schema),
      "strict": True,
      "schema": schema.model_json_schema(),
    },
  }

In [ ]:
model = "openrouter/google/gemini-2.5-flash-lite-preview-09-2025"


In [ ]:
response = litellm_completion(
  model=model,
  messages=[{"role": "user", "content": "Merhaba. Nasılsın?"}],
)
response.choices[0].message.content


In [ ]:
response = litellm_completion(
  model=model,
  messages=[create_message("Hello, how are you?")],
  response_format=_Response,
)


In [ ]:
_Response.model_validate_json(response.choices[0].message.content)


In [ ]:
# | export
class LLM:
  """A class for interacting with an LLM."""

  def __init__(
    self,
    model_name: str,  # The name of the model to use
    system_prompt: str = "",  # The system prompt to use.
    think: Optional[bool] = None,  # Whether to enable thinking mode
    output_schema: Optional[
      type[BaseModel]
    ] = None,  # Schema for structured output
  ):
    self.model_name = model_name
    self.system_prompt = system_prompt
    self.output_schema = output_schema
    self.messages: list[dict] = [create_message(system_prompt, "system")]
    self.think = think
    self.use_openrouter = "openrouter" in model_name

    if not self.use_openrouter:
      self._verify()

  def generate(self, message: str, **kwargs) -> str | OutputSchema:
    """One-shot generation. Does not maintain conversation history."""
    messages = [create_message(self.system_prompt, "system"), create_message(message, "user")]
    if self.use_openrouter:
      return self._cloud_call(messages, **kwargs)
    return self._local_call(messages, **kwargs)

  def chat(self, message: str, **kwargs) -> str | OutputSchema:
    """Multi-turn chat. Maintains conversation history."""
    self.messages.append(create_message(message, "user"))

    if self.use_openrouter:
      response = self._cloud_call(self.messages, **kwargs)
    else:
      response = self._local_call(self.messages, **kwargs)

    # Extract content for history: schema responses have .content, plain responses are strings
    if isinstance(response, BaseModel):
      self.messages.append(create_message(self._response_to_history_content(response), "assistant"))
    else:
      self.messages.append(create_message(response, "assistant"))
    return response

  def generate_with_resource(
    self,
    message: str,
    resource_path: Path | str,
    **kwargs,
  ) -> str | OutputSchema:
    """One-shot generation with an attached file (image/PDF).
    
    Only works with cloud models (openrouter). Does not maintain conversation history.
    
    Args:
      message: The text prompt to accompany the resource
      resource_path: Path to the file (image or PDF)
      **kwargs: Additional args passed to the API
      
    Returns:
      Model response as string or OutputSchema
    """
    if not self.use_openrouter:
      raise ValueError("generate_with_resource only works with cloud models (openrouter)")
    
    messages = [
      create_message(self.system_prompt, "system"),
      create_message_with_resource(message, resource_path, "user"),
    ]
    return self._cloud_call(messages, **kwargs)

  def _local_call(self, messages: list[dict], **kwargs) -> str | OutputSchema:
    """Call local Ollama model."""
    response: ChatResponse = ollama_chat(
      model=self.model_name,
      think=self.think,
      messages=messages,
      format=self.output_schema.model_json_schema() if self.output_schema else None,
      **kwargs,
    )
    return self._parse_response(response.message.content)

  def _cloud_call(self, messages: list[dict], **kwargs) -> str | OutputSchema:
    """Call cloud model."""
    return self._openrouter_call(messages, **kwargs)

  async def _acloud_call(self, messages: list[dict], **kwargs) -> str | OutputSchema:
    """Async cloud call."""
    return await self._aopenrouter_call(messages, **kwargs)

  def _build_openrouter_request(self, messages: list[dict], **kwargs) -> tuple[dict, dict]:
    api_key = os.getenv("OPENROUTER_API_KEY")
    if not api_key:
      raise RuntimeError("OPENROUTER_API_KEY is not configured")

    headers = {
      "Authorization": f"Bearer {api_key}",
      "Content-Type": "application/json",
    }
    http_referer = kwargs.pop("http_referer", None) or os.getenv("OPENROUTER_HTTP_REFERER")
    x_title = kwargs.pop("x_title", None) or os.getenv("OPENROUTER_X_TITLE")
    if http_referer:
      headers["HTTP-Referer"] = http_referer
    if x_title:
      headers["X-Title"] = x_title

    provider = dict(kwargs.pop("provider", {}) or {})
    if self.output_schema:
      provider.setdefault("require_parameters", True)

    payload = {
      "model": self.model_name.removeprefix("openrouter/"),
      "messages": messages,
      **kwargs,
    }
    if provider:
      payload["provider"] = provider

    if self.think is True and "reasoning" not in payload:
      payload["reasoning"] = {"effort": "low"}
    if self.output_schema:
      payload["response_format"] = _openrouter_response_format(self.output_schema)
      if not payload.get("stream"):
        plugins = list(payload.get("plugins") or [])
        if not any(plugin.get("id") == "response-healing" for plugin in plugins if isinstance(plugin, dict)):
          plugins.append({"id": "response-healing"})
        payload["plugins"] = plugins

    return headers, payload

  def _parse_openrouter_body(self, body: dict) -> str | OutputSchema:
    return self._parse_response(body["choices"][0]["message"]["content"])

  def _openrouter_call(self, messages: list[dict], **kwargs) -> str | OutputSchema:
    headers, payload = self._build_openrouter_request(messages, **kwargs)

    request = Request(
      "https://openrouter.ai/api/v1/chat/completions",
      data=json.dumps(payload).encode("utf-8"),
      headers=headers,
      method="POST",
    )
    with urlopen(request, timeout=120) as response:
      body = json.load(response)
    return self._parse_openrouter_body(body)

  async def _aopenrouter_call(self, messages: list[dict], **kwargs) -> str | OutputSchema:
    headers, payload = self._build_openrouter_request(messages, **kwargs)
    async with httpx.AsyncClient(timeout=120.0) as client:
      response = await client.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers=headers,
        json=payload,
      )
      response.raise_for_status()
      body = response.json()
    return self._parse_openrouter_body(body)

  async def agenerate(self, message: str, **kwargs) -> str | OutputSchema:
    """Async one-shot generation. Does not maintain conversation history."""
    import asyncio

    messages = [create_message(self.system_prompt, "system"), create_message(message, "user")]
    if self.use_openrouter:
      return await self._acloud_call(messages, **kwargs)
    return await asyncio.to_thread(self._local_call, messages, **kwargs)

  def _parse_response(self, content) -> str | OutputSchema:
    """Parse raw response content, validating against schema if set."""
    assert content is not None, "No content was returned"
    if self.output_schema:
      if isinstance(content, str):
        return self.output_schema.model_validate_json(content)
      return self.output_schema.model_validate(content)
    if isinstance(content, str):
      return content
    return json.dumps(content, ensure_ascii=False)

  def _response_to_history_content(self, response: BaseModel) -> str:
    content = getattr(response, "content", None)
    if isinstance(content, str) and content:
      return content
    return response.model_dump_json()

  async def agenerate_batch(
    self,
    messages: list[str],
    concurrency: int = 10,
    show_progress: bool = True,
    **kwargs,
  ) -> list[str | OutputSchema]:
    """Batch async generation with concurrency control.

    Args:
        messages: List of input messages to process
        concurrency: Max parallel requests (default 10)
        show_progress: Show tqdm progress bar
        **kwargs: Additional args passed to the API (e.g., temperature)

    Returns:
        List of responses in same order as inputs
    """
    import asyncio
    from tqdm.asyncio import tqdm_asyncio

    semaphore = asyncio.Semaphore(concurrency)

    async def process_one(message: str) -> str | OutputSchema:
      async with semaphore:
        return await self.agenerate(message, **kwargs)

    tasks = [process_one(msg) for msg in messages]
    if show_progress:
      return await tqdm_asyncio.gather(*tasks, desc="Processing")
    return await asyncio.gather(*tasks)

  def _verify(self):
    """Verify the model is available, download if missing."""
    if self.model_name not in list_local_models():
      _download_model(self.model_name)

# Tests


In [ ]:
# Test create_message with different roles
assert create_message("hello") == {"role": "user", "content": "hello"}
assert create_message("hello", "assistant") == {"role": "assistant", "content": "hello"}
assert create_message("system prompt", "system") == {"role": "system", "content": "system prompt"}
assert create_message(None) == {"role": "user", "content": None}
print("✓ create_message tests passed")


In [ ]:
# Test OutputSchema validation
class TestSchema(OutputSchema):
  sentiment: str

# Valid JSON should parse correctly
parsed = TestSchema.model_validate_json('{"content": "hello", "sentiment": "positive"}')
assert parsed.content == "hello"
assert parsed.sentiment == "positive"

# content field is optional
no_content = TestSchema.model_validate_json('{"sentiment": "positive"}')
assert no_content.content is None
assert no_content.sentiment == "positive"

print("✓ OutputSchema tests passed")

In [ ]:
# Test LLM initialization and routing logic
# OpenRouter models should set use_openrouter=True
llm_cloud = LLM("openrouter/google/gemini-2.5-flash", system_prompt="Be helpful")
assert llm_cloud.use_openrouter == True
assert llm_cloud.model_name == "openrouter/google/gemini-2.5-flash"
assert llm_cloud.system_prompt == "Be helpful"
assert llm_cloud.messages == [{"role": "system", "content": "Be helpful"}]

print("✓ LLM initialization tests passed")


In [ ]:
# Test list_local_models returns a list of strings
models = list_local_models()
assert isinstance(models, list)
assert all(isinstance(m, str) for m in models)
print(f"✓ list_local_models tests passed (found {len(models)} models)")


In [ ]:
# Test LLM._parse_response without schema (returns string as-is)
llm_no_schema = LLM("openrouter/test", output_schema=None)
assert llm_no_schema._parse_response("hello world") == "hello world"

# Test LLM._parse_response with schema
llm_with_schema = LLM("openrouter/test", output_schema=TestSchema)
result = llm_with_schema._parse_response('{"content": "test", "sentiment": "neutral"}')
assert isinstance(result, TestSchema)
assert result.content == "test"
assert result.sentiment == "neutral"

print("✓ LLM._parse_response tests passed")


In [ ]:
# Test agenerate_batch works with local Ollama models
import asyncio
async def test_agenerate_batch_local():
  llm_local = LLM("gemma3:27b")
  results = await llm_local.agenerate_batch(["Say 'test' and nothing else"], show_progress=False)
  assert isinstance(results, list)
  assert len(results) == 1
  return True

assert asyncio.run(test_agenerate_batch_local())
print("✓ agenerate_batch local model test passed")


In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()
